In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ISLP import load_data
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
College = load_data('College')
College.head()

,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


In [7]:
College['Private'] = College['Private'].map({'Yes': 1, 'No': 0})


X = College.drop(columns=['Apps']).values
y = College['Apps'].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

In [9]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)
mse_lin = mean_squared_error(y_test, y_pred_lin)
print(f"(b) Least squares test MSE: {mse_lin:.2f}")

(b) Least squares test MSE: 642753.90


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

alphas = np.logspace(-3, 5, 100)
ridge_cv = GridSearchCV(
    Ridge(), param_grid={'alpha': alphas}, cv=5, scoring='neg_mean_squared_error'
)
ridge_cv.fit(X_train_scaled, y_train)
best_alpha_ridge = ridge_cv.best_params_['alpha']
y_pred_ridge = ridge_cv.predict(X_test_scaled)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
print(f"(c) Ridge test MSE (best alpha={best_alpha_ridge:.4f}): {mse_ridge:.2f}")

(c) Ridge test MSE (best alpha=0.0010): 642755.04


In [11]:
lasso_cv = GridSearchCV(
    Lasso(max_iter=10000), param_grid={'alpha': alphas}, cv=5, scoring='neg_mean_squared_error'
)
lasso_cv.fit(X_train_scaled, y_train)
best_alpha_lasso = lasso_cv.best_params_['alpha']
y_pred_lasso = lasso_cv.predict(X_test_scaled)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
print(f"(d) Lasso test MSE (best alpha={best_alpha_lasso:.4f}): {mse_lasso:.2f}")

(d) Lasso test MSE (best alpha=9.1116): 652642.74


In [12]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('linreg', LinearRegression())
])

n_features = X_train.shape[1]
param_grid = {'pca__n_components': range(1, n_features + 1)}

pcr_cv = GridSearchCV(
    pipeline, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error'
)
pcr_cv.fit(X_train, y_train)
best_M = pcr_cv.best_params_['pca__n_components']
y_pred_pcr = pcr_cv.predict(X_test)
mse_pcr = mean_squared_error(y_test, y_pred_pcr)
print(f"(e) PCR test MSE (best M={best_M}): {mse_pcr:.2f}")

(e) PCR test MSE (best M=17): 642753.90
